# TOD-SciBERT — Google Colab
**Domain-Adaptive Pre-Training for Transit-Oriented Development Research**

### Before running:
1. Upload `tod_scibert/` folder to Google Drive → `MyDrive/TOD_DAPT/model/tod_scibert/`
2. Upload `dapt_corpus.txt` to Google Drive → `MyDrive/TOD_DAPT/corpus/dapt_corpus.txt`
3. Runtime → Change runtime type → **T4 GPU**

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/TOD_DAPT'
MODEL_PATH  = f'{BASE}/model/tod_scibert'
CORPUS_PATH = f'{BASE}/corpus/dapt_corpus.txt'

# Verify files exist
print('Model exists :', os.path.exists(MODEL_PATH))
print('Corpus exists:', os.path.exists(CORPUS_PATH))

## Step 2 — Install Dependencies

In [ ]:
!pip install transformers datasets accelerate pymupdf nltk python-docx -q
print('Done!')

## Step 3 — Clone Scripts from GitHub

In [ ]:
# Replace with your actual GitHub repo URL after pushing
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/tod-dapt.git'

!git clone {GITHUB_REPO} /content/tod-dapt
import sys
sys.path.insert(0, '/content/tod-dapt/scripts')
print('Scripts ready!')

## Step 4 — Perplexity Comparison (SciBERT vs TOD-SciBERT)

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SCIBERT_PATH = 'allenai/scibert_scivocab_uncased'
TOD_PATH     = MODEL_PATH
SEED         = 42
N_SENTENCES  = 5000
MLM_PROB     = 0.15

# Load corpus
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    all_sents = [l.strip() for l in f if len(l.strip()) > 40]

rng = np.random.default_rng(SEED)
sample = rng.choice(all_sents, size=min(N_SENTENCES, len(all_sents)), replace=False).tolist()
print(f'Sample sentences: {len(sample):,}')

In [ ]:
def compute_ppl(model_path, sentences, device, batch_size=32):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForMaskedLM.from_pretrained(model_path).to(device).eval()
    total_loss, total_tokens = 0.0, 0
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc = tok(batch, return_tensors='pt', padding=True,
                  truncation=True, max_length=256).to(device)
        labels = enc['input_ids'].clone()
        rand   = torch.rand(labels.shape, device=device)
        mask   = (rand < 0.15) & (labels != tok.pad_token_id)
        labels[~mask] = -100
        enc['input_ids'][mask] = tok.mask_token_id
        with torch.no_grad():
            out = mdl(**enc, labels=labels)
        n = mask.sum().item()
        total_loss   += out.loss.item() * n
        total_tokens += n
    avg_loss = total_loss / total_tokens
    return avg_loss, np.exp(avg_loss)

print('Computing SciBERT perplexity...')
sci_loss, sci_ppl = compute_ppl(SCIBERT_PATH, sample, device)
print(f'  Loss={sci_loss:.4f}  PPL={sci_ppl:.2f}')

print('Computing TOD-SciBERT perplexity...')
tod_loss, tod_ppl = compute_ppl(TOD_PATH, sample, device)
print(f'  Loss={tod_loss:.4f}  PPL={tod_ppl:.2f}')

print(f'\nPerplexity reduction: {(sci_ppl - tod_ppl)/sci_ppl*100:.1f}%')

## Step 5 — Retrieval Comparison (10 TOD Aspects)

In [ ]:
from transformers import AutoModel

ASPECTS = {
    'Land Use Mix'  : 'mixed land use transit oriented development',
    'Walkability'   : 'pedestrian walkability street design sidewalk',
    'Transit Access': 'public transit station access bus rail frequency',
    'Density'       : 'residential commercial density floor area ratio',
    'Parking'       : 'parking reduction car dependency vehicle ownership',
    'Cycling'       : 'cycling bicycle infrastructure network lane',
    'Green Space'   : 'green space park open space environment',
    'Affordability' : 'affordable housing income socioeconomic residents',
    'Station Area'  : 'station area development TOD zone catchment',
    'Urban Design'  : 'urban design placemaking street quality aesthetics',
}

def mean_pool(out, mask):
    m = mask.unsqueeze(-1).expand(out.size()).float()
    return (out * m).sum(1) / m.sum(1).clamp(min=1e-9)

def embed(model, tok, texts, device, bs=64):
    import numpy as np
    all_emb = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], padding=True, truncation=True,
                  max_length=256, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**enc)
        e = mean_pool(out.last_hidden_state, enc['attention_mask'])
        e = torch.nn.functional.normalize(e, p=2, dim=1)
        all_emb.append(e.cpu().numpy())
    return np.vstack(all_emb)

# Load retrieval models
sci_tok = AutoTokenizer.from_pretrained(SCIBERT_PATH)
sci_mod = AutoModel.from_pretrained(SCIBERT_PATH).to(device).eval()
tod_tok = AutoTokenizer.from_pretrained(TOD_PATH)
tod_mod = AutoModel.from_pretrained(TOD_PATH).to(device).eval()

# Use first 20K sentences for retrieval
ret_sents = all_sents[:20000]
print('Embedding corpus for retrieval...')
sci_embs = embed(sci_mod, sci_tok, ret_sents, device)
tod_embs = embed(tod_mod, tod_tok, ret_sents, device)
print('Done!')

In [ ]:
import pandas as pd
TOP_K = 15
rows  = []

for aspect, query in ASPECTS.items():
    sq = embed(sci_mod, sci_tok, [query], device)[0]
    tq = embed(tod_mod, tod_tok, [query], device)[0]
    sci_scores = sci_embs @ sq
    tod_scores = tod_embs @ tq
    sci_top = np.argsort(sci_scores)[::-1][:TOP_K]
    tod_top = np.argsort(tod_scores)[::-1][:TOP_K]
    sci_avg = float(sci_scores[sci_top].mean())
    tod_avg = float(tod_scores[tod_top].mean())
    rows.append({'Aspect': aspect, 'SciBERT': round(sci_avg,4),
                 'TOD-SciBERT': round(tod_avg,4),
                 'Improvement': round(tod_avg - sci_avg, 4)})

df = pd.DataFrame(rows)
avg_row = pd.DataFrame([{'Aspect':'OVERALL',
                         'SciBERT': round(df.SciBERT.mean(),4),
                         'TOD-SciBERT': round(df['TOD-SciBERT'].mean(),4),
                         'Improvement': round(df.Improvement.mean(),4)}])
df = pd.concat([df, avg_row], ignore_index=True)
print(df.to_string(index=False))

## Step 6 — t-SNE Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE

TOD_TERMS = [
    'transit-oriented development','TOD','station area','transit station',
    'rail station','walkability','pedestrian','cycling','bicycle lane',
    'public transit','bus rapid transit','mixed-use','land use mix',
    'density','floor area ratio','zoning','sustainable development',
    'green space','affordable housing','urban design','carbon emission',
    'energy efficiency','parking reduction','car dependency',
]
NON_TOD_TERMS = [
    'protein structure','cell biology','DNA sequence','enzyme activity',
    'neural network','chemical reaction','molecular weight',
    'organic compound','spectroscopy','quantum mechanics',
    'electromagnetic field','thermodynamics','particle physics',
    'machine learning','deep learning','algorithm complexity','data structure',
]
ALL_TERMS = TOD_TERMS + NON_TOD_TERMS
LABELS    = ['TOD']*len(TOD_TERMS) + ['Non-TOD']*len(NON_TOD_TERMS)

def get_cls_embs(model, tok, texts, device):
    model.eval()
    embs = []
    for t in texts:
        enc = tok(t, return_tensors='pt', truncation=True,
                  max_length=64, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc)
        embs.append(out.last_hidden_state[:,0,:].squeeze().cpu().numpy())
    return np.array(embs)

sci_emb2 = get_cls_embs(sci_mod, sci_tok, ALL_TERMS, device)
tod_emb2 = get_cls_embs(tod_mod, tod_tok, ALL_TERMS, device)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('t-SNE: SciBERT vs TOD-SciBERT', fontsize=14, fontweight='bold')

for ax, embs, title in [
    (axes[0], sci_emb2, 'SciBERT (Base)'),
    (axes[1], tod_emb2, 'TOD-SciBERT (Domain Adapted)'),
]:
    coords = TSNE(n_components=2, perplexity=10, random_state=42,
                  max_iter=2000, learning_rate='auto', init='pca').fit_transform(embs)
    tod_idx    = [i for i,l in enumerate(LABELS) if l=='TOD']
    nontod_idx = [i for i,l in enumerate(LABELS) if l=='Non-TOD']
    ax.scatter(coords[nontod_idx,0], coords[nontod_idx,1],
               c='#95A5A6', s=80, alpha=0.7, edgecolors='white')
    ax.scatter(coords[tod_idx,0], coords[tod_idx,1],
               c='#E74C3C', s=100, alpha=0.9, edgecolors='white')
    for i in tod_idx:
        ax.annotate(ALL_TERMS[i], coords[i], fontsize=6.5, alpha=0.85,
                    xytext=(3,3), textcoords='offset points')
    tod_c  = coords[tod_idx]
    spread = np.sqrt(((tod_c - tod_c.mean(0))**2).sum(1)).mean()
    ax.set_title(f'{title}\nSpread={spread:.1f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#f9f9f9')

plt.tight_layout()

# Save to Drive
out_dir = f'{BASE}/figures'
os.makedirs(out_dir, exist_ok=True)
plt.savefig(f'{out_dir}/tsne_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved to Drive!')

## Step 7 — Generate Expert Survey (Statements)

In [ ]:
# Run the survey generator script
import subprocess, sys

# Patch paths for Colab environment
survey_script = '/content/tod-dapt/scripts/14_generate_survey.py'

# Override paths inside the script
import importlib.util, types

# Or just re-run the key logic inline:
print('Survey generation: run cells above then call the survey function.')
print('Survey output will be saved to:', f'{BASE}/survey/')

## Step 8 — Download Outputs

In [ ]:
from google.colab import files
import os

# Download t-SNE figure
tsne_path = f'{BASE}/figures/tsne_comparison.png'
if os.path.exists(tsne_path):
    files.download(tsne_path)
    print('Downloaded: tsne_comparison.png')

# Download survey
survey_path = f'{BASE}/survey/expert_survey.docx'
if os.path.exists(survey_path):
    files.download(survey_path)
    print('Downloaded: expert_survey.docx')